In [1]:
%matplotlib tk

import pandas as pd
import numpy as np
import matplotlib.pylab as plt

csv_file_name = 'initial_walk_test_10-08-2026_16-38-16_long_walk.csv'
csv_path = '../data/measured_walks/'
csv_save_path = '../data/orientation_mahony/'
df = pd.read_csv(f'{csv_path}{csv_file_name}')

In [2]:
def mahony_filter(ax, ay, az, gx, gy, gz, dt_array, Kp=1.0, Ki=0.0):
    # initial quaternion (flat and forward)
    q = np.array([1.0, 0.0, 0.0, 0.0])

    quaternions = np.zeros((len(dt_array), 4))
    quaternions[0] = q

    gx_rad = np.deg2rad(gx)
    gy_rad = np.deg2rad(gy)
    gz_rad = np.deg2rad(gz)

    # Integral error terms (used to correct permanent hardware bias)
    eInt_x, eInt_y, eInt_z = 0.0, 0.0, 0.0

    for i in range(1, len(dt_array)):
        dt = dt_array[i]

        wx = gx_rad[i]
        wy = gy_rad[i]
        wz = gz_rad[i]

        # normalize accelerometer readings
        a_x, a_y, a_z = ax[i], ay[i], az[i]
        acc_norm = np.sqrt(a_x**2 + a_y**2 + a_z**2)
        
        # Only correct if accel is valid (not in 0g freefall)
        if acc_norm > 0.0:
            a_x /= acc_norm
            a_y /= acc_norm
            a_z /= acc_norm

            qw, qx, qy, qz = q

            # Estimate gravity direction mathematically using current quaternion
            # This calculates what a perfect accelerometer WOULD read right now
            v_x = 2.0 * (qx * qz - qw * qy)
            v_y = 2.0 * (qw * qx + qy * qz)
            v_z = 1.0 - 2.0 * (qx**2 + qy**2)

            # Calculate error between measured gravity and estimated gravity (Cross Product)
            # [image-comments/Screenshot_2026-08-15_140539.png]
            e_x = (a_y * v_z) - (a_z * v_y)
            e_y = (a_z * v_x) - (a_x * v_z)
            e_z = (a_x * v_y) - (a_y * v_x)

            # Integral Error: Accumulate error over time
            if Ki > 0.0:
                eInt_x += e_x * dt
                eInt_y += e_y * dt
                eInt_z += e_z * dt
            else:
                eInt_x, eInt_y, eInt_z = 0.0, 0.0, 0.0

            # PI Controller: Nudge the raw gyro towards the accelerometer's truth
            wx += (Kp * e_x) + (Ki * eInt_x)
            wy += (Kp * e_y) + (Ki * eInt_y)
            wz += (Kp * e_z) + (Ki * eInt_z)

        # Proceed with normal Gyro Integration using the CORRECTED speeds
        qw, qx, qy, qz = q

        q_dot_w = 0.5 * (-qx*wx - qy*wy - qz*wz)
        q_dot_x = 0.5 * ( qw*wx + qy*wz - qz*wy)
        q_dot_y = 0.5 * ( qw*wy - qx*wz + qz*wx)
        q_dot_z = 0.5 * ( qw*wz + qx*wy - qy*wx)

        q[0] += q_dot_w * dt
        q[1] += q_dot_x * dt
        q[2] += q_dot_y * dt
        q[3] += q_dot_z * dt

        q = q / np.linalg.norm(q)
        quaternions[i] = q

    return quaternions
   


def quaternions_to_euler(quats):
    # Slice into four separate 1D arrays for w, x, y, z
    w = quats[:, 0]
    x = quats[:, 1]
    y = quats[:, 2]
    z = quats[:, 3]
    
    # Roll (X axis rotation)
    sinr_cosp = 2 * (w * x + y * z)
    cosr_cosp = 1 - 2 * (x**2 + y**2)
    roll = np.arctan2(sinr_cosp, cosr_cosp) 
    
    # Pitch (Y axis rotation)
    sinp = 2 * (w * y - z * x)
    sinp = np.clip(sinp, -1.0, 1.0) 
    pitch = np.arcsin(sinp)
    
    # Yaw (Z axis rotation)
    siny_cosp = 2 * (w * z + x * y)
    cosy_cosp = 1 - 2 * (y**2 + z**2)
    yaw = np.arctan2(siny_cosp, cosy_cosp)
    
    # Convert from radians back to degrees
    return np.rad2deg(roll), np.rad2deg(pitch), np.rad2deg(yaw)


In [ ]:
# Calculate the TRUE dt for every single sample in seconds.
dt_array = df['t_us'].diff().fillna(5000).to_numpy() / 1e6

# Grab Gyro and Accel data
gx, gy, gz = df['gx'].to_numpy(), df['gy'].to_numpy(), df['gz'].to_numpy()
ax, ay, az = df['ax'].to_numpy(), df['ay'].to_numpy(), df['az'].to_numpy()

# Kp=1.0 is the "trust factor". 1.0 means we trust the accelerometer a moderate amount.
quaternions = mahony_filter(ax, ay, az, gx, gy, gz, dt_array, Kp=1.0, Ki=0.0)

# convert the 4D quaternions into 3D Euler angles (degrees)
roll, pitch, yaw = quaternions_to_euler(quaternions)

time_sec = (df['t_us'] - df['t_us'].iloc[0]) / 1e6

fig, axs = plt.subplots(figsize=(12, 6))

# We dont plot the yaw as gravity provides zero information about our heading. Yaw
# is mathematically unobservable to an accelerometer. The Mahony filter cannot fix its drift.
axs.plot(time_sec, roll, label='Roll (X axis tilt)', color='tab:blue', linewidth=1.5)
axs.plot(time_sec, pitch, label='Pitch (Y axis tilt)', color='tab:orange', linewidth=1.5)

axs.set_title("Ungated Mahony Filter Implementation (Also active in non ZVWs)")
axs.set_xlabel("Time (Seconds)")
axs.set_ylabel("Angle (Degrees)")
axs.legend(loc='upper left')
axs.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.savefig(f'{csv_save_path}{csv_file_name}'.replace('.csv', '_ungated_mahony_filter.png'), dpi=120)
plt.show()